# 🎵 Music to MIDI — Google Colab (Free GPU)

在 Colab 免费 T4 GPU 上运行 Music to MIDI，通过 Gradio 公开链接访问。

**使用方法：**
1. 点击上方菜单 `Runtime → Change runtime type → T4 GPU`
2. 依次运行下面的单元格
3. 最后一个单元格会输出一个公开链接，打开即可使用

In [ ]:
#@title 1️⃣ 检查 GPU 是否可用
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU: {gpu_name} ({vram:.1f} GB VRAM)")
else:
    print("❌ 未检测到 GPU！请点击 Runtime → Change runtime type → T4 GPU")
    raise RuntimeError("需要 GPU 运行时")

In [ ]:
#@title 2️⃣ 克隆仓库 & 安装依赖
import os

# 克隆项目
if not os.path.exists("/content/music-to-midi"):
    !git clone https://github.com/mason369/music-to-midi.git /content/music-to-midi
else:
    print("仓库已存在，跳过克隆")

os.chdir("/content/music-to-midi")

# 安装依赖（跳过 PyQt6 和 spaces，Colab 不需要）
!pip install -q \
    torchaudio \
    lightning einops 'transformers==4.45.1' mir-eval deprecated smart-open \
    nest-asyncio scipy scikit-learn torchmetrics wandb \
    huggingface_hub mido librosa soundfile \
    'audio-separator>=0.36.0' onnxruntime \
    'gradio>=4.44.0' \
    tqdm pyyaml psutil numba

# 安装 FFmpeg（音频解码需要）
!apt-get install -y -qq ffmpeg > /dev/null 2>&1

print("\n✅ 依赖安装完成")

In [ ]:
#@title 3️⃣ 下载 YourMT3+ 源码 & 模型权重
import os, sys
os.chdir("/content/music-to-midi")

# 下载 YourMT3 源码（从 HF Space 获取 amt/src）
yourmt3_dir = "/content/music-to-midi/space/YourMT3"
amt_src = os.path.join(yourmt3_dir, "amt", "src")

if not os.path.exists(os.path.join(amt_src, "model", "ymt3.py")):
    print("正在下载 YourMT3 源码...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        "mimbres/YourMT3",
        repo_type="space",
        local_dir=yourmt3_dir,
        allow_patterns=["amt/src/**"],
        ignore_patterns=["*.ckpt", "*.bin", "*.safetensors", "amt/logs/**"],
    )
    print("✅ YourMT3 源码下载完成")
else:
    print("✅ YourMT3 源码已存在")

# 创建符号链接：项目根/YourMT3 → space/YourMT3
# YourMT3Transcriber._load_model() 会搜索项目根下的 YourMT3/
root_link = "/content/music-to-midi/YourMT3"
if not os.path.exists(root_link):
    os.symlink(yourmt3_dir, root_link)
    print(f"✅ 创建符号链接: YourMT3 → space/YourMT3")

# 将 amt/src 加入 sys.path
if amt_src not in sys.path:
    sys.path.insert(0, amt_src)

# 下载模型权重
print("\n正在下载模型权重（约 800MB，首次需要等待）...")
sys.path.insert(0, "/content/music-to-midi")
from download_sota_models import download_ultimate_moe
download_ultimate_moe()
print("\n✅ 模型权重就绪")

In [ ]:
#@title 4️⃣ 启动 Gradio 应用（会生成公开链接）
import os, sys, tempfile, logging, signal
from pathlib import Path

os.chdir("/content/music-to-midi")

# ── 杀掉旧 Gradio 进程（避免端口占用）──
try:
    import subprocess
    result = subprocess.run(
        ["fuser", "-k", "7860/tcp"],
        capture_output=True, timeout=5
    )
except Exception:
    pass

# ── 路径设置 ──
PROJECT_ROOT = "/content/music-to-midi"
amt_src = "/content/music-to-midi/space/YourMT3/amt/src"
for p in [PROJECT_ROOT, amt_src]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── 环境变量 ──
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_MIN_LOG_LEVEL"] = "3"
os.environ["NUMBA_CACHE_DIR"] = "/tmp/numba_cache"
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

# ── 日志 ──
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("colab-app")

# ── 验证 YourMT3 可导入 ──
try:
    from model.ymt3 import YourMT3  # noqa: F401
    logger.info("✅ YourMT3 代码可用")
except ImportError as e:
    raise RuntimeError(f"YourMT3 导入失败: {e}，请重新运行第 3 步")

# ── 导入核心模块 ──
from src.models.data_models import Config, ProcessingStage
from src.core.pipeline import MusicToMidiPipeline
from src.utils.gpu_utils import clear_gpu_memory, get_device

import gradio as gr

# ── 设备信息 ──
import torch
device_label = f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU"
logger.info(f"当前设备: {device_label}")

# ── 日志文件（供前端轮询）──
LOG_FILE = "/tmp/midi_process.log"

class _RobustFileHandler(logging.Handler):
    def __init__(self, filename):
        super().__init__()
        self.filename = filename
    def emit(self, record):
        try:
            msg = self.format(record)
            with open(self.filename, "a", encoding="utf-8") as f:
                f.write(msg + "\n")
        except Exception:
            pass

_fh = _RobustFileHandler(LOG_FILE)
_fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S"))
_fh.setLevel(logging.INFO)
for _name in ("colab-app", "src.core", "src.utils"):
    logging.getLogger(_name).addHandler(_fh)

def clear_logs():
    try:
        open(LOG_FILE, "w").close()
    except Exception:
        pass

def read_logs():
    try:
        with open(LOG_FILE, "r", encoding="utf-8", errors="replace") as f:
            content = f.read()
        content = content.replace("\x00", "")
        lines = content.strip().split("\n")
        return "\n".join(lines[-50:]) if lines and lines[0] else ""
    except Exception as e:
        return f"[error] {e}"

# ── 核心转换函数 ──
def convert_audio_to_midi(audio_path, mode, quality, progress=gr.Progress()):
    if audio_path is None:
        raise gr.Error("请先上传音频文件")

    clear_logs()
    logger.info(f"音频文件: {Path(audio_path).name}")
    logger.info(f"处理模式: {mode}")
    logger.info(f"转写质量: {quality}")

    config = Config()
    config.processing_mode = "vocal_split" if mode == "人声分离 + 分别转写" else "smart"
    config.transcription_quality = quality

    pipeline = MusicToMidiPipeline(config)
    output_dir = tempfile.mkdtemp(prefix="midi_output_")

    def on_progress(p):
        stage_name = {
            ProcessingStage.PREPROCESSING: "预处理",
            ProcessingStage.SEPARATION: "音源分离",
            ProcessingStage.TRANSCRIPTION: "音频转写",
            ProcessingStage.VOCAL_TRANSCRIPTION: "人声转写",
            ProcessingStage.SYNTHESIS: "MIDI合成",
            ProcessingStage.COMPLETE: "完成",
        }.get(p.stage, str(p.stage))
        progress(p.overall_progress, desc=f"[{stage_name}] {p.message}")

    try:
        result = pipeline.process(audio_path=audio_path, output_dir=output_dir, progress_callback=on_progress)
    except Exception as e:
        logger.error(f"转换失败: {e}")
        raise gr.Error(f"转换失败: {e}")
    finally:
        try:
            clear_gpu_memory()
        except Exception:
            pass

    output_files = []
    if result.midi_path and Path(result.midi_path).exists():
        output_files.append(result.midi_path)
    if result.vocal_midi_path and Path(result.vocal_midi_path).exists():
        output_files.append(result.vocal_midi_path)
    if result.accompaniment_midi_path and Path(result.accompaniment_midi_path).exists():
        if result.accompaniment_midi_path != result.midi_path:
            output_files.append(result.accompaniment_midi_path)
    # 分离后的音频文件（人声 + 伴奏 WAV）
    if result.separated_audio:
        for audio_key in ("vocals", "no_vocals"):
            audio_file = result.separated_audio.get(audio_key)
            if audio_file and Path(audio_file).exists():
                output_files.append(audio_file)

    bpm_str = f"{result.beat_info.bpm:.1f}" if result.beat_info else "N/A"
    status = "\n".join([
        "--- 转换完成 ---",
        f"耗时: {result.processing_time:.1f} 秒",
        f"总音符数: {result.total_notes}",
        f"BPM: {bpm_str}",
        f"设备: {device_label}",
    ])
    return output_files, status

def update_mode_info(mode):
    if mode == "人声分离 + 分别转写":
        return "**BS-RoFormer + YourMT3+** — 先分离人声与伴奏，再分别转写，输出两个 MIDI 文件。"
    return "**YourMT3+ MoE** — 直接对完整音频进行多乐器转写，精确识别 128 种 GM 乐器。"

# ── JavaScript 日志轮询（通过 head 注入，兼容 share 链接）──
LOG_POLL_JS = """<script>
(function() {
    var pollCount = 0;
    var _pollTimer = setInterval(function() {
        pollCount++;
        var ta = document.querySelector('.log-box textarea');
        if (!ta) return;
        var setter = Object.getOwnPropertyDescriptor(
            HTMLTextAreaElement.prototype, 'value'
        ).set;
        fetch('./api/read_logs', {
            method: 'POST',
            headers: {'Content-Type': 'application/json'},
            body: JSON.stringify({data: []})
        })
        .then(function(r) { return r.json(); })
        .then(function(json) {
            var logText = (json.data && json.data[0]) ? json.data[0] : '';
            if (logText) {
                setter.call(ta, logText);
            }
            ta.dispatchEvent(new Event('input', {bubbles: true}));
            ta.scrollTop = ta.scrollHeight;
        })
        .catch(function() {});
    }, 2000);
})();
</script>"""

# ── Gradio 界面 ──
with gr.Blocks(
    title="Music to MIDI (Colab GPU)",
    head=LOG_POLL_JS,
    theme=gr.themes.Base(primary_hue=gr.themes.colors.blue, neutral_hue=gr.themes.colors.slate),
) as demo:
    gr.Markdown("# 🎵 Music to MIDI\n在 Google Colab 免费 GPU 上运行 — 基于 YourMT3+ MoE 深度学习模型")

    with gr.Row():
        with gr.Column(scale=5):
            audio_input = gr.Audio(label="上传音频文件", type="filepath")
            gr.Markdown("<small>支持 MP3, WAV, FLAC, OGG, M4A</small>")
            mode_radio = gr.Radio(["YourMT3+ 多乐器转写", "人声分离 + 分别转写"], value="YourMT3+ 多乐器转写", label="处理模式")
            mode_info = gr.Markdown("**YourMT3+ MoE** — 直接对完整音频进行多乐器转写，精确识别 128 种 GM 乐器。")
            mode_radio.change(fn=update_mode_info, inputs=[mode_radio], outputs=[mode_info], api_name=False)
            quality_radio = gr.Radio(["fast", "balanced", "best"], value="balanced", label="转写质量")
            convert_btn = gr.Button("▶ 开始转换", variant="primary", size="lg")
            gr.Markdown(f"当前设备: **{device_label}**")

        with gr.Column(scale=5):
            status_output = gr.Textbox(label="状态", interactive=False, lines=6, placeholder="等待转换...")
            file_output = gr.File(label="下载文件", file_count="multiple")
            log_output = gr.Textbox(label="控制台日志", interactive=False, lines=12, max_lines=20, elem_classes="log-box")

    convert_btn.click(fn=convert_audio_to_midi, inputs=[audio_input, mode_radio, quality_radio], outputs=[file_output, status_output])

    # 日志 API 端点（供 JavaScript 轮询调用）
    _log_poll_btn = gr.Button(visible=False)
    _log_poll_btn.click(
        fn=read_logs,
        inputs=[],
        outputs=[log_output],
        api_name="read_logs",
        queue=False,
    )

    gr.Markdown("<center>基于 [YourMT3+](https://github.com/mimbres/YourMT3) MoE 模型 | [GitHub](https://github.com/mason369/music-to-midi)</center>")

print("\n" + "=" * 60)
print("🚀 启动中... 公开链接将在下方显示")
print("=" * 60 + "\n")
demo.launch(share=True, server_name="0.0.0.0", server_port=7860)